# Import

In [86]:
from telethon.sync import TelegramClient
import re
import csv
from datetime import datetime
import asyncio
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup


# Telegram

In [ ]:
# Конфигурация (заполните своими данными)
api_id = '16494191'
api_hash = 'b450c865395daf03c0853c6b095ab324'
channel_username = 'tondexscreenerupdates'

# Уточненное регулярное выражение для TON-адресов
ton_address_pattern = r'\bEQ[a-zA-Z0-9_-]{43,48}\b'

async def parse_channel():
    async with TelegramClient('session_name', api_id, api_hash) as client:
        channel = await client.get_entity(channel_username)
        
        messages = client.iter_messages(
            channel,
            # offset_date=datetime(2025,12,31),
            # reverse=True,
            # search='новый токен'
        )

        results = []
        cnt = 0
        async for message in messages:
            cnt+=1
            # Проверка наличия текста и даты
            if not message.text or message.date.year not in [2024, 2025]:
                continue
            
            # Приведение к строке и обработка ключевых слов
            text = str(message.text)
            if 'Contract' not in text and 'TON' not in text:
                continue
                
            # Поиск адресов с обработкой ошибок
            # text1 = text[text.find('**contract**:'):]
            # print(text1[text1.find('`')+1:text1.find('\n')-1])
            try:
                addresses = re.findall(ton_address_pattern, text)
            except TypeError:
                continue
                
            # print('yes')
            # break
            if addresses:
                results.append({
                    'date': message.date.strftime("%Y-%m-%d %H:%M:%S"),
                    'addresses': addresses[0],
                    'text': message.text[:100] + '...'
                })
        print(len(results), cnt)
        # Сохранение результатов
        with open(f'ton_{channel_username}.csv', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=['date', 'addresses', 'text'])
            writer.writeheader()
            writer.writerows(results)

# Универсальный запуск
try:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(parse_channel())
except Exception as e:
    print(f"Ошибка выполнения: {str(e)}")


519 577


In [83]:
adf=pd.read_csv(f'ton_{channel_username}.csv')
adf[adf['addresses'].apply(lambda x: len(x)) == 48].to_csv(f'ton_{channel_username}.csv', index=False)

# Memelandia

In [ ]:


# Настройка headless-режима браузера
options = Options()
options.add_argument("--headless")
driver = webdriver.Chrome(options=options)

# Открываем страницу Memelandia
driver.get("https://ton.org/ru/memelandia")
time.sleep(5)  # Ждём загрузки таблицы (можно увеличить при медленном интернете)

# Находим таблицу мемкоинов (пример: ищем все строки таблицы)
rows = driver.find_elements(By.CSS_SELECTOR, "table tr")

memecoins = []
for row in rows[1:]:  # Пропускаем заголовок таблицы
    cells = row.find_elements(By.TAG_NAME, "td")
    if len(cells) >= 2:
        name = cells[0].text
        ticker = cells[1].text
        memecoins.append({"name": name, "ticker": ticker})

driver.quit()

for coin in memecoins:
    print(coin)


# CoinGecko

In [ ]:

url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "category": "ton-meme-coins",
    "order": "market_cap_desc",
    "per_page": 250,
    "page": 1
}

memecoins = []
while True:
    response = requests.get(url, params=params)
    if not response.ok:
        print(f'Press F: {response.reason}')
        break
    data = response.json()
    if not data:
        break
    for coin in data:
        while True:
            print(coin)
            coin_url = f"https://api.coingecko.com/api/v3/coins/{coin['id']}"
            coin_response = requests.get(coin_url)
            if not coin_response.ok:
                print(f"error with {coin['id']}")
                time.sleep(10)
                continue
            else:
                memecoins.append({
                    "name": coin["name"],
                    "ticker": coin["symbol"],
                    "image": coin["image"],
                    "contract_address": coin_response.json()["contract_address"]
                })
                break
    params["page"] += 1

print(f"Найдено мемкоинов: {len(memecoins)}")


{'id': 'notcoin', 'symbol': 'not', 'name': 'Notcoin', 'image': 'https://coin-images.coingecko.com/coins/images/33453/large/rFmThDiD_400x400.jpg?1701876350', 'current_price': 0.00284361, 'market_cap': 290945665, 'market_cap_rank': 240, 'fully_diluted_valuation': 290945665, 'total_volume': 93039166, 'high_24h': 0.00302718, 'low_24h': 0.00276712, 'price_change_24h': -2.5119029401124e-05, 'price_change_percentage_24h': -0.87561, 'market_cap_change_24h': -2782370.4157372713, 'market_cap_change_percentage_24h': -0.94726, 'circulating_supply': 102456956923.5639, 'total_supply': 102456956923.5639, 'max_supply': None, 'ath': 0.02836145, 'ath_change_percentage': -90.01227, 'ath_date': '2024-06-02T18:00:38.587Z', 'atl': 0.00160895, 'atl_change_percentage': 76.05677, 'atl_date': '2025-04-16T18:01:30.613Z', 'roi': None, 'last_updated': '2025-05-11T08:14:48.366Z'}
{'id': 'dogs-2', 'symbol': 'dogs', 'name': 'Dogs', 'image': 'https://coin-images.coingecko.com/coins/images/39699/large/dogs_logo_200x200

In [77]:
df = pd.DataFrame(memecoins)
df = df[df['contract_address'] != '0x15ac90165f8b45a80534228bdcb124a011f62fee']

In [78]:
df.to_csv('memcoins_coingecko.csv')

# sdfa

In [85]:

url = "https://grafun.io/launches"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

memecoins = []
for card in soup.select(".launch-card"):
    address = card.select_one(".contract-address").text.strip()
    memecoins.append(address)

print(memecoins)


[]


In [87]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = Options()
options.add_argument("--headless")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
driver = webdriver.Chrome(options=options)

try:
    driver.get("https://gaspump.tg/#/")
    time.sleep(10)  # Ожидание загрузки SPA

    # Перехват сетевых запросов
    performance_log = driver.get_log('performance')
    memecoin_addresses = set()

    # Поиск адресов в логах сети
    for entry in performance_log:
        if "getLaunchpads" in str(entry):  # Ищем эндпоинты API
            response = driver.execute_cdp_cmd('Network.getResponseBody', {'requestId': entry['message']['params']['requestId']})
            if response['body']:
                for item in response['body'].get('data', []):
                    if item.get('jettonAddress'):
                        memecoin_addresses.add(item['jettonAddress'])

    # Альтернативно: парсинг интерфейса
    launchpad_cards = WebDriverWait(driver, 20).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".launchpad-card"))
    )
    for card in launchpad_cards:
        address = card.find_element(By.CSS_SELECTOR, ".contract-address").text
        if address.startswith("EQ"):
            memecoin_addresses.add(address)

    print(f"Найдено адресов: {len(memecoin_addresses)}")
    print(list(memecoin_addresses)[:10])  # Пример первых 10 адресов

except Exception as e:
    print(f"Ошибка: {e}")

finally:
    driver.quit()


Ошибка: Message: invalid argument: log type 'performance' not found
  (Session info: chrome=136.0.7103.93)
Stacktrace:
	GetHandleVerifier [0x00007FF60792CF25+75717]
	GetHandleVerifier [0x00007FF60792CF80+75808]
	(No symbol) [0x00007FF6076F8F9A]
	(No symbol) [0x00007FF60778FAD7]
	(No symbol) [0x00007FF607777340]
	(No symbol) [0x00007FF607740421]
	(No symbol) [0x00007FF6077411B3]
	GetHandleVerifier [0x00007FF607C2D6FD+3223453]
	GetHandleVerifier [0x00007FF607C27CA2+3200322]
	GetHandleVerifier [0x00007FF607C45AD3+3322739]
	GetHandleVerifier [0x00007FF6079469FA+180890]
	GetHandleVerifier [0x00007FF60794E0FF+211359]
	GetHandleVerifier [0x00007FF607935274+109332]
	GetHandleVerifier [0x00007FF607935422+109762]
	GetHandleVerifier [0x00007FF60791BA39+4825]
	BaseThreadInitThunk [0x00007FFE1AC37374+20]
	RtlUserThreadStart [0x00007FFE1BFFCC91+33]

